# Create King Faisal Prize Awards (PRIZE PATTERN)

King Faisal Prize laureates from the official King Faisal Prize all-winners table and official detail pages.

**Awarding body:** King Faisal Foundation - F4320323301, ROR `https://ror.org/00n60x364`, DOI `10.13039/501100005418`.

**Prerequisites:**
- Run `scripts/local/king_faisal_prize_to_s3.py` first.
- Admin must upload the generated parquet because contractor access has no S3 credentials.

**Data sources:**
- `https://kingfaisalprize.org/all-winners/` for official laureate rows
- `https://kingfaisalprize.org/nominations/` for the official SAR 750,000 / US$200,000 prize-components rule
- Official per-laureate detail pages under `https://kingfaisalprize.org/...`

**S3:** `s3a://openalex-ingest/awards/king_faisal_prize/king_faisal_prize_laureates.parquet`

**Mapping notes:**
- Prize pattern: one row per `(King Faisal Prize category x laureate)`.
- `lead_investigator` is the official laureate. Organizations keep `given_name` NULL and store the organization name in `family_name`.
- The source table's `country` column is an official laureate country/nationality-style field, not an affiliation country. It is retained in raw data but intentionally not mapped into `lead_investigator.affiliation.country`.
- Official current prize total is SAR 750,000 per prize category/year. Amount is split by the official winner count for the same `(award_year, award_category)` so shared prizes do not multiply the prize total.
- `currency` is SAR from the official nominations/prize-components page.
- Provenance is `king_faisal_prize`.
- Priority is 87; priorities 66-86 are reserved by in-flight fork PRs, with Hewlett at 86.
- No `jobs/create_funder_sourced_awards.yaml` task is added here. Admin should upload/run/QA first, then wire the job after verification.

## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.king_faisal_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/king_faisal_prize/king_faisal_prize_laureates.parquet`;

## Step 1.5: Inspect raw data before transform

In [ ]:
%sql
SELECT COUNT(*) AS raw_rows FROM openalex.awards.king_faisal_prize_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.king_faisal_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.king_faisal_prize_raw LIMIT 10;

In [ ]:
%sql
-- Required money-field scan before mapping.
SELECT column_name
FROM (DESCRIBE openalex.awards.king_faisal_prize_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|portion|winner_count';

In [ ]:
%sql
-- Required currency-field scan before mapping.
SELECT column_name
FROM (DESCRIBE openalex.awards.king_faisal_prize_raw)
WHERE LOWER(column_name) RLIKE 'currenc|currency|ccy|iso_4217';

In [ ]:
%sql
SELECT
    award_category,
    COUNT(*) AS rows,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year,
    COUNT(topic) AS rows_with_topic,
    COUNT(citation) AS rows_with_citation,
    COUNT(landing_page_url) AS rows_with_landing_page,
    COUNT(laureate_given_name) AS rows_with_given_name,
    COUNT(laureate_family_name) AS rows_with_family_name
FROM openalex.awards.king_faisal_prize_raw
GROUP BY award_category
ORDER BY award_category;

In [ ]:
%sql
-- Sanity-check the source money columns. The script verifies the official
-- SAR 750,000 prize-components page before writing parquet and splits that
-- total by winner_count for shared category/year prizes.
SELECT
    COUNT(*) AS total_rows,
    COUNT(source_prize_amount_sar) AS rows_with_total_amount,
    COUNT(source_award_amount) AS rows_with_apportioned_amount,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies,
    MIN(TRY_CAST(source_prize_amount_sar AS DOUBLE)) AS min_total_amount,
    MAX(TRY_CAST(source_prize_amount_sar AS DOUBLE)) AS max_total_amount,
    AVG(TRY_CAST(source_prize_amount_sar AS DOUBLE)) AS avg_total_amount,
    MIN(TRY_CAST(source_award_amount AS DOUBLE)) AS min_apportioned_amount,
    MAX(TRY_CAST(source_award_amount AS DOUBLE)) AS max_apportioned_amount,
    AVG(TRY_CAST(source_award_amount AS DOUBLE)) AS avg_apportioned_amount,
    MIN(TRY_CAST(winner_count AS DOUBLE)) AS min_winner_count,
    MAX(TRY_CAST(winner_count AS DOUBLE)) AS max_winner_count
FROM openalex.awards.king_faisal_prize_raw;

In [ ]:
%sql
-- Source ID uniqueness check. This must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.king_faisal_prize_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

## Step 1.6: Funder existence fail-fast

In [ ]:
%sql
-- Must return exactly 1 row. If 0 rows, STOP and do not run the transform.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320323301;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.king_faisal_prize_awards
USING delta
AS
WITH king_faisal_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323301
),
normalized AS (
    SELECT
        k.*,
        TRY_CAST(k.award_year AS INT) AS parsed_award_year,
        TRY_CAST(k.source_award_amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(k.source_prize_amount_sar AS DOUBLE) AS parsed_total_amount,
        TRY_CAST(k.winner_count AS DOUBLE) AS parsed_winner_count
    FROM openalex.awards.king_faisal_prize_raw k
),
award_rows AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':king_faisal_prize:', LOWER(TRIM(k.funder_award_id))))) % 9000000000 AS id,
        CONCAT(k.prize_title, ' ', CAST(k.parsed_award_year AS STRING), ' - ', k.laureate_name) AS display_name,
        COALESCE(NULLIF(k.citation, ''), NULLIF(k.meta_description, ''), NULLIF(k.quote, ''), NULLIF(k.topic, '')) AS description,
        f.funder_id,
        k.funder_award_id AS funder_award_id,
        k.parsed_amount AS amount,
        NULLIF(k.currency, '') AS currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name, f.ror_id, f.doi
        ) AS funder,
        'prize' AS funding_type,
        k.prize_title AS funder_scheme,
        'king_faisal_prize' AS provenance,
        TRY_TO_DATE(CONCAT(CAST(k.parsed_award_year AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(CAST(k.parsed_award_year AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        k.parsed_award_year AS start_year,
        k.parsed_award_year AS end_year,
        struct(
            NULLIF(k.laureate_given_name, '') AS given_name,
            NULLIF(k.laureate_family_name, '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        NULLIF(k.landing_page_url, '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized k
    CROSS JOIN king_faisal_funder f
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    concat('https://api.openalex.org/works?filter=awards.id:G', id) AS works_api_url,
    created_date,
    updated_date
FROM award_rows;

## Step 3: Insert into openalex_awards_raw at priority 87

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'king_faisal_prize' AND priority = 87;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       87 AS priority
FROM openalex.awards.king_faisal_prize_awards;

## Verification

In [ ]:
%sql
-- Step 6.7 amount/currency check. Expect 100% SAR amount coverage,
-- with values split from the official SAR 750,000 category/year total.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.king_faisal_prize_awards;

In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 120) AS display_name,
    funder_scheme,
    amount,
    currency,
    start_year,
    lead_investigator.given_name,
    lead_investigator.family_name,
    landing_page_url
FROM openalex.awards.king_faisal_prize_awards
ORDER BY start_year DESC, funder_scheme, lead_investigator.family_name
LIMIT 25;

In [ ]:
%sql
SELECT id, COUNT(*) AS dupes
FROM openalex.awards.king_faisal_prize_awards
GROUP BY id
HAVING COUNT(*) > 1;

In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS dupes
FROM openalex.awards.king_faisal_prize_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year,
    SUM(amount) AS summed_amount
FROM openalex.awards.king_faisal_prize_awards
GROUP BY funder_scheme
ORDER BY funder_scheme;

In [ ]:
%sql
-- Data completeness (runbook Step 6.3 form, adapted for prize pattern).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    COUNT(lead_investigator.given_name) AS has_given_name,
    COUNT(lead_investigator.family_name) AS has_family_name,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates
FROM openalex.awards.king_faisal_prize_awards;

In [ ]:
%sql
SELECT funder_id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.king_faisal_prize_awards
GROUP BY funder_id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.king_faisal_prize_awards
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
SELECT provenance, priority, COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'king_faisal_prize' AND priority = 87
GROUP BY provenance, priority;